---
# Clinical Data Quality Engine (DQIE)
# Notebook 03 — Anomaly Detection
# Purpose: Detect value, temporal, relational, and source anomalies in the Silver Layer
---


# Preparations
---

## Setup do ambiente

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# Add project root to PYTHONPATH
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PYTHONPATH OK:", PROJECT_ROOT)

## Imports of anomaly detection modules

In [ ]:
from src._4_anomalies.detect_value_anomalies import detect_value_anomalies
from src._4_anomalies.detect_temporal_anomalies import detect_temporal_anomalies
from src._4_anomalies.detect_relational_anomalies import detect_relational_anomalies
from src._4_anomalies.detect_source_anomalies import detect_source_anomalies

# Load Silver Layer
---

In [ ]:
df_patients = pd.read_parquet("../data/_2_silver/patients.parquet")
df_injuries = pd.read_parquet("../data/_2_silver/injuries.parquet")
df_sessions = pd.read_parquet("../data/_2_silver/sessions.parquet")
df_clinical = pd.read_parquet("../data/_2_silver/clinical_reports.parquet")
df_ocr_json = pd.read_parquet("../data/_2_silver/ocr_extracted.parquet")
df_ocr_images = pd.read_parquet("../data/_2_silver/ocr_images.parquet")

print("Silver Layer loaded.")

# Value Anomalies
---

In [ ]:
print("\n## Anomaly Detection — Value Anomalies")
value_anomalies = detect_value_anomalies(df_sessions)

if len(value_anomalies) > 0:
    print(f"Value anomalies detected: {len(value_anomalies)}")
    display(value_anomalies)
else:
    print("No value anomalies detected.")

# Temporal Anomalies
---

In [ ]:
print("\n## Anomaly Detection — Temporal Anomalies")
temporal_anomalies = detect_temporal_anomalies(
    patients=df_patients,
    injuries=df_injuries,
    sessions=df_sessions,
    clinical_reports=df_clinical
)

if len(temporal_anomalies) > 0:
    print(f"Temporal anomalies detected: {len(temporal_anomalies)}")
    display(temporal_anomalies)
else:
    print("No temporal anomalies detected.")

# Relational Anomalies
---

In [ ]:
print("\n## Anomaly Detection — Relational Anomalies")
relational_anomalies = detect_relational_anomalies(
    patients=df_patients,
    injuries=df_injuries,
    sessions=df_sessions,
    clinical_reports=df_clinical,
    ocr_reports=df_ocr_json
)


if len(relational_anomalies) > 0:
    print(f"Relational anomalies detected: {len(relational_anomalies)}")
    display(relational_anomalies)
else:
    print("No relational anomalies detected.")

# Source Anomalies
---

In [ ]:
print("\n## Anomaly Detection — Source Anomalies")
source_anomalies = detect_source_anomalies(
    csv_data=df_sessions,
    sql_data=df_sessions,
    ocr_text=df_ocr_json,
    ocr_images=df_ocr_images,
    clinical_json=df_clinical
)

if len(source_anomalies) > 0:
    print(f"Source anomalies detected: {len(source_anomalies)}")
    display(source_anomalies)
else:
    print("No source anomalies detected.")

# Consolidation
---

In [ ]:
print("\n## Consolidating anomalies...")

all_anomalies = pd.concat([
    value_anomalies,
    temporal_anomalies,
    relational_anomalies,
    source_anomalies
], ignore_index=True)

print(f"Total anomalies detected: {len(all_anomalies)}")
display(all_anomalies)

# Summary
---

In [ ]:
print("\n--- ANOMALY SUMMARY ---")
print(f"Value anomalies: {len(value_anomalies)}")
print(f"Temporal anomalies: {len(temporal_anomalies)}")
print(f"Relational anomalies: {len(relational_anomalies)}")
print(f"Source anomalies: {len(source_anomalies)}")
print(f"TOTAL anomalies: {len(all_anomalies)}")

if len(all_anomalies) == 0:
    print("Dataset is clean. Proceed to Notebook 04 — Reconciliation.")
else:
    print("Anomalies detected. Proceed to Notebook 04 — Reconciliation.")

# Save anomaly outputs to Gold Layer

In [ ]:
def normalize(df):
    for col in df.columns:
        df[col] = df[col].astype("string")
    return df

value_anomalies = normalize(value_anomalies)
temporal_anomalies = normalize(temporal_anomalies)
relational_anomalies = normalize(relational_anomalies)
source_anomalies = normalize(source_anomalies)

In [ ]:
output_dir = "../data/_3_gold/anomalies/"
os.makedirs(output_dir, exist_ok=True)

value_anomalies.to_parquet(f"{output_dir}/value_anomalies.parquet", index=False)
temporal_anomalies.to_parquet(f"{output_dir}/temporal_anomalies.parquet", index=False)
relational_anomalies.to_parquet(f"{output_dir}/relational_anomalies.parquet", index=False)
source_anomalies.to_parquet(f"{output_dir}/source_anomalies.parquet", index=False)

print("Anomaly outputs saved to Gold Layer.")
